get historical data from yfinance

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

Calculate sigma on twenty day window preceding the start of our five-day window on which we will calculate portfolio returns. Bin obtained sigmas with tiny bin size (equate all sigmas in a bin with the midpoint of the bin). 

In [ ]:
tickers = [
    "SPY",
    "VTI",
    "VEA",
    "IWM",
    "VWO",
    "VGT",
    "VHT",
    "IBB",
    "XBI",
    "QQQ",
    "PDBC",
    "VGK",
    "FBT",
    "ARKG",
    "BBH",
    "IBIT",
    "FBTC",
    "GBTC",
    "ETHA",
    "BTC",
    "VNQ",
    "SCHH",
    "XLRE",
    "IYR",
    "USRT",
    "ICF",
    "RWR",
    "SMH",
    "SOXX",
    "SOXL",
    "NVDL",
    "USD",
    "XSD",
    "GRID",
    "NLR",
    "ICLN",
    "TAN",
    "FTGC",
    "HGER",
]  # stopped at technology on https://etfdb.com/

ticker = "SPY"
start_date, end_date = "2016-03-01", "2026-03-01"
df = yf.download(tickers, start=start_date, end=end_date)["Close"]

[*********************100%***********************]  39 of 39 completed


In [ ]:
window_length = 20
res_list = []

for leverage_factor in np.linspace(1.01, 3, 100):
    for ticker in tickers:

        data = df[[ticker]]
        data["log_returns"] = np.log(data[ticker] / data[ticker].shift(1))
        data["realized_vol"] = data[["log_returns"]].rolling(
            window_length
        ).std() * np.sqrt(252)
        data["leveraged_log_returns"] = np.log(
            leverage_factor * (np.exp(data["log_returns"]) - 1) + 1
        )
        data["PnL"] = np.exp(
            data["leveraged_log_returns"].shift(-5).rolling(5).sum()
        ) - np.exp(data["log_returns"].shift(-5).rolling(5).sum())

        data["leverage_factor"] = leverage_factor

        res_list.append(
            data.dropna().reset_index()[["leverage_factor", "realized_vol", "PnL"]]
        )  # address issue with Ticker column of indices

res_df = pd.concat(res_list)

res_df.to_parquet(f"output/historical_data/lev_sigma_std.parquet")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
